# 04 · 选哪些训练图给剪枝打分？

在 CASIA-WebFace 训练的同一个 Dense-Aug ResNet-18 上，六种方法从 3,071 张训练图中选择 calibration photos。OAGP 与 OAGP-max 共用选出的图片和逐图梯度。主问题是用 1,024 张图打分时，选图方式能否改善四种留出区域遮挡下的 LFW 人脸验证准确率。

直接 **Run All** 可复算公开结果。复算只需仓库里的配对相似度和标签。最后一节可使用原始数据和 checkpoint 重新剪枝、推理。

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

marker = Path("results/pruning/calibration_selection/paired_seed_metrics.csv")
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / marker).is_file()), None)
if root is None and "google.colab" in sys.modules:
    root = Path("/content/CS260D_Final_Project")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yhr04/CS260D_Final_Project.git", str(root)], check=True)
if root is None or not (root / marker).is_file():
    raise FileNotFoundError("Open this notebook inside a checkout containing the calibration selection results.")

study = root / marker.parent
metrics = pd.read_csv(root / marker)
control = pd.read_csv(study.parent / "one_shot_seed42" / "metrics.csv")
selectors = {
    "random": "Random", "coreset": "Facility location", "kcenter": "K-center",
    "greedi": "GreeDi", "greedy_merge": "Greedy-Merge", "sieve": "Sieve-Streaming",
}
methods = {"alpha_weighted": "OAGP", "condition_max": "OAGP-max"}

## 1. 固定预算下的选图效果

每个 seed 都有一半 clean、一半 square-occluded 训练图，每张图算一次 Taylor gradient。各层分别归一化两种条件的分数，OAGP 以 α = 0.5 加权平均，OAGP-max 逐权重取较大值。两者在除首层外的 19 个卷积层各剪 50%，总计保留 5,578,752 个可剪权重，微调 0 步。

每种遮挡都测试相同的 6,000 对脸，只遮住第二张。准确率指模型正确判断两张脸是否同一人的比例。每折只用另外九折拟合阈值。主指标对眼部、口鼻、上半脸、下半脸四项准确率等权平均，再对打分 seed 42、43、44 取均值。Δ 是与同一 seed 的随机选图之差，单位为百分点（pp）。

In [ ]:
primary = metrics.query("budget == 1024 and metric == 'region_mean'")
summary = primary.pivot_table(index="strategy", columns="method", values="accuracy_pct").loc[list(selectors)]
deltas = primary.pivot_table(index="strategy", columns="method", values="minus_random_pp").loc[list(selectors)]
table = pd.DataFrame(index=summary.index)
for key, label in methods.items():
    table[f"{label} (%)"] = summary[key]
    table[f"{label} Δ random (pp)"] = deltas[key]
display(table.rename(index=selectors).round(3))

regions = ["eyes", "mouth_nose", "upper_half", "lower_half"]
controls = control.query("dataset == 'lfw' and scoring_seed == 42 and method in ['dense', 'magnitude']")
region_controls = controls[controls.condition.isin(regions)].groupby("method").accuracy.mean() * 100
square_controls = controls.query("condition == 'square50'").set_index("method").accuracy * 100
display(pd.DataFrame({"Four regions (%)": region_controls, "Square 50% (%)": square_controls}).round(3))

在主设置下，Sieve-Streaming 的四区域均值为 OAGP 92.97%、OAGP-max 93.87%。在相同稀疏度下，Magnitude 达到 94.93%，并且无需 calibration 梯度。Square 50% 上，OAGP-max 搭配 Sieve 为 89.51%，Magnitude 为 89.17%。这项更重的方块遮挡也曾参与 CASIA 验证集上的 checkpoint 选择。

## 2. 更换预算和测试条件

修改 `metric` 可查看各预算下的 clean、square10、square30、square50、eyes、mouth_nose、upper_half、lower_half、seen_square_mean 或 region_mean。`seen_square_mean` 对 square10 和 square30 等权平均。三个 seed 共用同一 checkpoint、候选池与测试对。Sieve 和 GreeDi 会随预算重新选图。

In [ ]:
metric = "region_mean"
if metric not in set(metrics.metric):
    raise ValueError(f"Choose one of {sorted(metrics.metric.unique())}")
by_budget = metrics[metrics.metric == metric].pivot_table(
    index=["budget", "method"], columns="strategy", values="accuracy_pct",
)
display(by_budget[list(selectors)].rename(columns=selectors).rename(index=methods, level="method").round(3))

## 3. 从配对相似度复核结果

下面的独立 NumPy 脚本重新拟合 10 折阈值，将 108 个剪枝配置、8 种测试条件的 8,640 个逐折正确对数与存档逐项比较，并核对 CSV 中 1,080 行准确率和配对差。原始照片、权重和 GPU 均可省去。

In [ ]:
verification = subprocess.run(
    [sys.executable, str(root / "scripts/verify_calibration_selection.py")],
    check=True, capture_output=True, text=True,
)
print(verification.stdout, end="")

## 4. 可选 · 从训练图重新剪枝和推理

如需重新运行神经网络，先按项目 README 下载 Dense-Aug 权重，再用 notebook 01 从相同的 CASIA 与 LFW 源文件制作缓存。程序会核对 checkpoint、候选图像、标签、split 和测试遮挡。 [项目 README](../README.md) · [数据准备 notebook](01_data.ipynb)

将 `RUN_PRUNING` 设为 `True`。默认示例重做一次 seed 42、B = 1,024 的 Sieve 选图对应的两种剪枝结果。`replay` 使用公开的选图顺序。`recompute` 重新提取 clean embeddings 并运行所选算法。不同设备的浮点结果可能改变排序或个别配对判定，输出会列出与记录相同的排名数和四区域差值。如需完整网格，设置 `seeds=(42, 43, 44)`、`budgets=(256, 512, 1024)`、`strategies=tuple(selectors)`。

In [ ]:
RUN_PRUNING = False
if RUN_PRUNING:
    sys.path.insert(0, str(root / "src"))
    from occprune.calibration_ablation import rerun_calibration

    reruns = rerun_calibration(
        checkpoint="checkpoints/dense_aug_seed42.pt",
        cache_dir="data/cache",
        selection_mode="replay",
        seeds=(42,), budgets=(1024,), strategies=("sieve",), device="auto",
    )
    recorded = metrics.query("metric == 'region_mean'").set_index(["strategy", "seed", "budget", "method"])
    for trial in reruns:
        key = (trial["strategy"], trial["seed"], trial["budget"])
        print(key, "matching ranks", trial["ordered_ranks_matching_recorded"], "/", trial["budget"])
        for name, result in trial["methods"].items():
            observed = result["region_mean_pct"]
            difference = observed - recorded.at[(*key, name), "accuracy_pct"]
            print(f"{methods[name]}  {observed:.3f}%  ({difference:+.3f} pp vs. recorded)")